In [4]:
import sys
from pathlib import Path

notebook_dir = Path().resolve()
example_notebooks_dir = notebook_dir.parent.parent
bat_autobidding_dir = example_notebooks_dir.parent

sys.path.insert(0, str(example_notebooks_dir))
sys.path.insert(0, str(bat_autobidding_dir))

In [5]:
import pandas as pd
import optuna
import pickle
from functools import partial
from pathlib import Path

from simulator.simulation.modules import Campaign
from simulator.simulation.utils_visualization import data_prep_vis, plot_history_article
from simulator.simulation.simulate import simulate_campaign
from simulator.validation.check_results import autobidder_check

In [6]:
from simulator.model.rlb_dp_bidder import RLBDPBidder

In [7]:
from experiments.exp_configs import RND42N10Config

config = RND42N10Config()
config.ensure_artifact_dirs()

In [8]:
model_name = 'rlb_dp'

In [9]:
auction_mode = config.auction_mode
best_params_subfolder = config.experiment_name
# metric to optimize: CPC_REL / RMSE / SCR
metric = config.metric
random_seed = config.random_seed
n_trials = config.n_trials
data_config = config.data_config

In [21]:
artifacts_subpath = f'{model_name}_{metric.lower()}_{auction_mode}'

In [11]:
stats_path = data_config['train']['stats_path']
campaigns_path = data_config['train']['campaigns_path']

In [12]:
stats_df = pd.read_csv(stats_path)

In [13]:
def objective_rlb_dp(trial, metric='RMSE_T', auction_mode='FPA'):

    max_bid = trial.suggest_float('max_bid', 10, 500, log=True)
    # maybe better here to use relative_value? like max_bid as daily/budget_share, budget_share
    gamma = trial.suggest_float('gamma', 0.80, 1.00) 
    N_bound = trial.suggest_int('N_bound', 6, 72)
    B_bound = trial.suggest_int('B_bound', 1e3, 2e4, log=True)

    custom_params = {
        "max_bid": max_bid,
        "gamma": gamma,
        "model_path": None,
        "N_bound": N_bound,
        "B_bound": B_bound,
    }


    bidder = RLBDPBidder(custom_params)

    bidder.fit(stats_df)
    import os, uuid

    os.makedirs("tmp_models", exist_ok=True)
    TMP_MODEL_PATH = f"tmp_models/rlb_dp_trial{trial.number}_{uuid.uuid4().hex}.pkl"
    bidder.save_model(TMP_MODEL_PATH)

    # прогоняем через тот же пайплайн проверки
    res = autobidder_check(
        bidder=RLBDPBidder,
        params={
            "input_campaigns": campaigns_path,
            "input_stats": stats_path,
            "max_bid": max_bid,
            "gamma": gamma,
            "model_path": TMP_MODEL_PATH,
            "N_bound": N_bound,
            "B_bound": B_bound,
        },
        auction_mode=auction_mode,
    )

    print(f"CPC_REL: {res['score'][0]}, rmse: {res['score'][1]}, SCR: {res['score'][2]}")
    if metric == 'RMSE_T':
        return res['score'][1]
    elif metric == 'CPC_REL':
        return res['score'][0]
    elif metric == 'SCR':
        return res['score'][2]


def opt_search_rlb_dp(n_trials, metric='RMSE_T', auction_mode='FPA'):
    study = optuna.create_study(
        direction='maximize' if metric == 'SCR' else 'minimize',
        sampler=optuna.samplers.TPESampler(seed=random_seed)
    )
    
    study.optimize(
        partial(objective_rlb_dp, metric=metric, auction_mode=auction_mode),
        n_trials=n_trials,
        n_jobs=6
    )

    print('Best trial:')
    trial = study.best_trial
    print(f'  Value: {trial.value}')
    print('  Params: ')

    dict_path = f'best_params/rlb_dp_{metric.lower()}_{auction_mode}.pkl'
    params_dict = {}
    for key, value in trial.params.items():
        print(f'    {key}: {value}')
        params_dict[key] = value

    with open(dict_path, 'wb') as f:
        pickle.dump(params_dict, f)

    return study


def train_best_rlb_dp(best_params_path, model_path='rlb_dp_model_tuned.pkl'):
    """Обучить и сохранить модель с лучшими параметрами (после optuna)."""
    with open(best_params_path, 'rb') as f:
        best_params = pickle.load(f)

    custom_params = {
        "max_bid": best_params["max_bid"],
        "gamma": best_params["gamma"],
        "model_path": None,
        "N_bound": best_params["N_bound"],
        "B_bound": best_params["B_bound"],
    }

    bidder = RLBDPBidder(custom_params)
    bidder.fit(stats_df)
    bidder.save_model(model_path)
    return bidder


In [14]:
study_rlb = opt_search_rlb_dp(n_trials, metric, auction_mode)

[I 2026-03-22 22:10:27,520] A new study created in memory with name: no-name-47b927f6-327a-4696-af4f-79250af8c6af
Hours:   0%|          | 0/54 [00:00<?, ?it/s]









Hours:  37%|███▋      | 20/54 [00:00<00:00, 197.13it/s]









Hours:  74%|███████▍  | 40/54 [00:01<00:00, 51.27it/s] 






Hours:  94%|█████████▍| 51/54 [00:01<00:00, 35.52it/s]


Hours: 100%|██████████| 40/40 [00:01<00:00, 35.57it/s] 











Hours: 100%|██████████| 54/54 [00:01<00:00, 30.18it/s]












Hours: 100%|██████████| 44/44 [00:01<00:00, 24.09it/s]
























Hours: 100%|██████████| 44/44 [00:02<00:00, 15.08it/s]














Hours: 100%|██████████| 71/71 [00:06<00:00, 11.09it/s]
[I 2026-03-22 22:21:48,294] Trial 1 finished with value: 31385.36120274262 and parameters: {'max_bid': 485.1341480914632, 'gamma': 0.901008142761115, 'N_bound': 64, 'B_bound': 4412}. Best is trial 1 with value: 31385.36120274262.


CPC_REL: 266.3565144416296, rmse: 1.5458781693626387, SCR: 31385.36120274262


Hours: 100%|██████████| 43/43 [00:00<00:00, 212.01it/s]
[I 2026-03-22 22:21:52,748] Trial 2 finished with value: 27179.139628512567 and parameters: {'max_bid': 30.91532988920657, 'gamma': 0.8967939711131678, 'N_bound': 40, 'B_bound': 1466}. Best is trial 1 with value: 31385.36120274262.
[I 2026-03-22 22:21:52,933] Trial 5 finished with value: 30109.945257758693 and parameters: {'max_bid': 52.10711038580981, 'gamma': 0.8168044662948563, 'N_bound': 44, 'B_bound': 5009}. Best is trial 1 with value: 31385.36120274262.


CPC_REL: 296.1741297060741, rmse: 1.1607912264397142, SCR: 27179.139628512567
CPC_REL: 272.95881052592136, rmse: 1.1427333718568424, SCR: 30109.945257758693


[I 2026-03-22 22:21:53,666] Trial 4 finished with value: 32734.828781577256 and parameters: {'max_bid': 224.23560705210977, 'gamma': 0.9520992172525834, 'N_bound': 44, 'B_bound': 3292}. Best is trial 4 with value: 32734.828781577256.


...

CPC_REL: 319.513118783789, rmse: 1.1661946596347734, SCR: 26140.87894109334


[I 2026-03-22 22:28:06,880] Trial 8 finished with value: 22083.007844862754 and parameters: {'max_bid': 12.54718317385639, 'gamma': 0.978906440788971, 'N_bound': 52, 'B_bound': 2919}. Best is trial 4 with value: 32734.828781577256.


CPC_REL: 428.4395176066186, rmse: 1.1956302931722362, SCR: 22083.007844862754


[I 2026-03-22 22:28:07,155] Trial 7 finished with value: 32123.098014504198 and parameters: {'max_bid': 87.33179914449809, 'gamma': 0.927946265025837, 'N_bound': 34, 'B_bound': 4891}. Best is trial 4 with value: 32734.828781577256.


CPC_REL: 265.3843231415613, rmse: 1.1505653095614203, SCR: 32123.098014504198
Best trial:
  Value: 32734.828781577256
  Params: 
    max_bid: 224.23560705210977
    gamma: 0.9520992172525834
    N_bound: 44
    B_bound: 3292


In [23]:
# best_params_path = f'best_params/{best_params_subfolder}/{metric.lower()}.pkl'
best_params_path=f'best_params/{artifacts_subpath}.pkl'
best_model_path = f'best_models/{artifacts_subpath}.pkl'

rlb_bidder_best = train_best_rlb_dp(best_params_path, best_model_path)

Hours: 100%|██████████| 44/44 [00:00<00:00, 119.21it/s]


> Eval

In [24]:
best_params_rlb = pd.read_pickle(best_params_path)
best_model_rlb_path = best_model_path

In [25]:
campaigns_path_test = data_config['test']['campaigns_path']
stats_path_test = data_config['test']['stats_path']

In [26]:
res = autobidder_check(
    bidder=RLBDPBidder,
    params = {
        "input_campaigns": campaigns_path_test,
        "input_stats": stats_path_test,
        "model_path": best_model_path,
        **best_params_rlb
    },
    auction_mode=auction_mode,
)

In [28]:
res['score']

(328.2458729550812, 1.2594471274394863, 32979.553049794, 0.026459143968871595)